# 🤖 Workshop Prático: Agentes Autônomos com Gemini API e Function Calling
**Rec'n'Play 2026 · Petros Barreto (GDG Recife)**

Este notebook tem a oficina inteira e roda no navegador, sem instalar nada.

**Antes de começar:**
1. Gere sua chave grátis em https://aistudio.google.com/apikey
2. No menu da esquerda do Colab, clique na **chave 🔑 (Secrets)** → *Adicionar novo secret* → nome `GEMINI_API_KEY`, valor = sua chave → ative o acesso do notebook.
3. Rode as células de cima para baixo com **Shift + Enter**.

In [ ]:
# Instala as bibliotecas (leva uns 30 segundos)
!pip install -q "google-genai>=2.3.0" pydantic requests python-dotenv

Criamos o arquivo `comum.py` com o cliente e as cores do terminal (o mesmo do repositório).

In [ ]:
%%writefile comum.py
"""
comum.py — utilidades compartilhadas por todos os exercícios.

Aqui ficam três coisas que todo script precisa:
  1. Carregar a chave da API do arquivo .env
  2. Criar o "cliente" (o objeto que conversa com a Gemini API)
  3. Funções para imprimir mensagens coloridas no terminal
"""
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

RAIZ = Path(__file__).resolve().parent
load_dotenv(RAIZ / ".env")

# Modelo padrão da oficina (pode ser trocado no .env)
MODELO = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")


def criar_cliente():
    """Cria o cliente da Gemini API usando a chave do .env."""
    from google import genai

    chave = os.getenv("GEMINI_API_KEY", "")
    if not chave:  # No Google Colab, a chave fica em "Secrets" (ícone de chave 🔑)
        try:
            from google.colab import userdata  # type: ignore

            chave = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    if not chave or chave == "cole_sua_chave_aqui":
        erro(
            "Chave da API não encontrada!\n"
            "   1. Gere uma chave em https://aistudio.google.com/apikey\n"
            "   2. Copie .env.example para .env e cole a chave lá"
        )
        sys.exit(1)
    return genai.Client(api_key=chave)


# ---------- Cores no terminal (códigos ANSI, sem bibliotecas extras) ----------
_CORES = {"azul": "34", "verde": "32", "amarelo": "33", "vermelho": "31", "cinza": "90", "magenta": "35"}


def cor(texto: str, nome: str) -> str:
    return f"\033[{_CORES[nome]}m{texto}\033[0m"


def titulo(texto: str):
    print("\n" + cor("━" * 60, "azul"))
    print(cor(f"  {texto}", "azul"))
    print(cor("━" * 60, "azul"))


def info(texto: str):
    print(cor("ℹ ", "azul") + texto)


def ok(texto: str):
    print(cor("✔ ", "verde") + texto)


def aviso(texto: str):
    print(cor("⚠ ", "amarelo") + texto)


def erro(texto: str):
    print(cor("✖ ", "vermelho") + texto)

In [ ]:
from comum import criar_cliente, MODELO
cliente = criar_cliente()
print("Pronto! Modelo:", MODELO)

---
## Módulo 1 — Primeira chamada
Um **cliente** conversa com a API. Passamos o **modelo** e o **input** (o prompt). A resposta é uma *interaction* com `id`, `steps` e `output_text`.

In [ ]:
interaction = cliente.interactions.create(
    model=MODELO,
    input="Explique em 3 frases, para quem nunca programou, o que é uma API.",
)
print(interaction.output_text)
print("\nPassos:", [p.type for p in interaction.steps])
print("ID:", interaction.id)

### Memória: o modelo não lembra sozinho
Usamos `previous_interaction_id` para continuar a conversa. `system_instruction` define o comportamento e precisa ir em **toda** chamada.

In [ ]:
INSTRUCOES = "Você é um monitor simpático do Rec'n'Play. Responda curto, em português."

t1 = cliente.interactions.create(model=MODELO, input="Oi! Meu nome é Ana e amo frevo.", system_instruction=INSTRUCOES)
print("Gemini:", t1.output_text)

t2 = cliente.interactions.create(model=MODELO, input="Qual é meu nome e do que eu gosto?",
                                 system_instruction=INSTRUCOES, previous_interaction_id=t1.id)
print("Gemini:", t2.output_text)

### Alucinação ao vivo
Perguntamos algo que o modelo **não tem como saber**. Observe se ele inventa.

In [ ]:
pergunta = "Qual oficina acontece às 14h na sala Maracatu do Rec'n'Play 2026?"
print("SEM regra:\n", cliente.interactions.create(model=MODELO, input=pergunta).output_text)
print("\nCOM regra de honestidade:\n", cliente.interactions.create(
    model=MODELO, input=pergunta,
    system_instruction="Se não tiver acesso ao dado, diga 'Não tenho essa informação'.").output_text)

---
## Módulo 2 — Structured outputs
Descrevemos o formato com **Pydantic** e exigimos com `response_format`. A API devolve JSON válido e o Pydantic confere os tipos.

✍️ **Mão na massa:** adicione o campo `precisa_notebook: bool` e rode de novo.

In [ ]:
from typing import List, Literal, Optional

from pydantic import BaseModel, Field

from comum import MODELO, criar_cliente, titulo, ok, cor


# 1) Descrevemos o FORMATO que queremos receber
class Palestrante(BaseModel):
    nome: str = Field(description="Nome completo da pessoa palestrante")
    cargo: Optional[str] = Field(default=None, description="Cargo ou empresa, se mencionado")


class Palestra(BaseModel):
    titulo: str = Field(description="Título da atividade")
    tipo: Literal["palestra", "oficina", "painel", "outro"] = Field(description="Formato da atividade")
    horario_inicio: str = Field(description="Horário de início no formato HH:MM (24h)")
    duracao_minutos: int = Field(description="Duração total em minutos")
    local: str = Field(description="Sala ou espaço onde acontece")
    palestrantes: List[Palestrante]
    publico_iniciante: bool = Field(description="True se o texto indica que iniciantes são bem-vindos")
    temas: List[str] = Field(description="De 2 a 5 palavras-chave sobre o conteúdo")


# 2) Um texto "do mundo real", escrito do jeito que as pessoas escrevem
TEXTO = """
Galeraaa!! 🚀 Amanhã tem oficina massa no Rec'n'Play: "Construindo agentes com Gemini"
com a Ana Beatriz (dev na Acme Tech) e o Carlos Lima. Começa 1 da tarde e vai até
às 5, lá na sala Frevo, 2º andar. Não precisa saber IA não, é pra quem tá começando!
Vai rolar function calling, APIs e structured outputs. Leva o notebook 💻
"""

cliente = criar_cliente()
titulo("Extraindo dados estruturados de um post informal")

# 3) Pedimos a extração passando o schema em response_format
interaction = cliente.interactions.create(
    model=MODELO,
    input=f"Extraia as informações da atividade descrita neste texto:\n{TEXTO}",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": Palestra.model_json_schema(),
    },
)

print(cor("JSON bruto devolvido pela API:", "cinza"))
print(interaction.output_text)

# 4) Validamos: se algo estiver fora do schema, o Pydantic acusa o erro aqui
palestra = Palestra.model_validate_json(interaction.output_text)

print()
ok("Agora é um objeto Python de verdade. Veja como é fácil usar:")
print(f"   Título:     {palestra.titulo}")
print(f"   Início:     {palestra.horario_inicio}  ({palestra.duracao_minutos} min)")
print(f"   Local:      {palestra.local}")
print(f"   Quem:       {', '.join(p.nome for p in palestra.palestrantes)}")
print(f"   Iniciante?  {'Sim' if palestra.publico_iniciante else 'Não'}")
print(f"   Temas:      {palestra.temas}")

### O modelo interpreta, o código decide

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

from comum import MODELO, criar_cliente, titulo, cor


class Triagem(BaseModel):
    sentimento: Literal["positivo", "neutro", "negativo"]
    categoria: Literal["conteudo", "infraestrutura", "comida", "acessibilidade", "outro"]
    urgente: bool = Field(description="True se exige ação da organização ainda hoje")
    resumo: str = Field(description="Resumo do feedback em até 12 palavras")


FEEDBACKS = [
    "A oficina de agentes foi incrível, saí sabendo fazer function calling!",
    "O wi-fi da sala Frevo caiu de novo e ninguém consegue instalar as bibliotecas.",
    "Não tem rampa de acesso no palco principal, meu amigo cadeirante não conseguiu subir.",
    "O café acabou cedo, mas de resto tudo ok.",
]

ACOES = {  # Regras de negócio: quem decide é o código
    "infraestrutura": "Abrir chamado para a equipe de TI",
    "acessibilidade": "Acionar a coordenação de acessibilidade",
    "comida": "Avisar a equipe de alimentação",
    "conteudo": "Enviar para a curadoria",
    "outro": "Registrar na planilha geral",
}

cliente = criar_cliente()
titulo("Triagem automática de feedbacks")

for texto in FEEDBACKS:
    resposta = cliente.interactions.create(
        model=MODELO,
        input=f"Classifique este feedback de um participante de evento: {texto}",
        response_format={"type": "text", "mime_type": "application/json", "schema": Triagem.model_json_schema()},
    )
    t = Triagem.model_validate_json(resposta.output_text)

    marcador = cor("🔴 URGENTE", "vermelho") if t.urgente else cor("🟢 normal ", "verde")
    print(f"\n{marcador} [{t.sentimento:>8}] {t.resumo}")
    print(cor(f"            → Ação automática: {ACOES[t.categoria]}", "cinza"))

---
## Módulo 3 — Function calling em 4 passos
1. **Declarar** a ferramenta · 2. **Pedir** (pergunta + ferramentas) · 3. **Executar** (nosso código) · 4. **Devolver** o resultado

> O modelo **nunca** executa nada. Ele só pede.

In [ ]:
import json

import requests

from comum import MODELO, criar_cliente, titulo, cor, info, ok


# ─────────────────────────────────────────────────────────────
# A função Python "de verdade". Ela não tem nada de IA.
# ─────────────────────────────────────────────────────────────
def previsao_do_tempo(cidade: str) -> dict:
    """Consulta o clima atual de uma cidade na API Open-Meteo."""
    # 1ª chamada: descobrir latitude/longitude da cidade
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": cidade, "count": 1, "language": "pt", "countryCode": "BR"},
        timeout=10,
    ).json()
    if not geo.get("results"):
        return {"erro": f"Cidade '{cidade}' não encontrada."}
    lugar = geo["results"][0]

    # 2ª chamada: clima atual naquela coordenada
    clima = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lugar["latitude"],
            "longitude": lugar["longitude"],
            "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
            "timezone": "America/Recife",
        },
        timeout=10,
    ).json()["current"]

    return {
        "cidade": lugar["name"],
        "estado": lugar.get("admin1"),
        "temperatura_c": clima["temperature_2m"],
        "umidade_pct": clima["relative_humidity_2m"],
        "chuva_mm": clima["precipitation"],
        "vento_kmh": clima["wind_speed_10m"],
        "horario_medicao": clima["time"],
    }


# ─────────────────────────────────────────────────────────────
# PASSO 1 — Declarar a ferramenta para o modelo.
# É como um "cardápio": o modelo lê a descrição para decidir quando usar.
# Descrições boas = decisões boas. Escreva como se fosse para um colega.
# ─────────────────────────────────────────────────────────────
DECLARACAO_CLIMA = {
    "type": "function",
    "name": "previsao_do_tempo",
    "description": (
        "Retorna o clima ATUAL (temperatura, umidade, chuva e vento) de uma cidade brasileira. "
        "Use sempre que o usuário perguntar sobre tempo, calor, chuva ou o que vestir."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "cidade": {"type": "string", "description": "Nome da cidade, ex.: Recife, Olinda, Caruaru"}
        },
        "required": ["cidade"],
    },
}

# Tabela que liga o NOME que o modelo pede à FUNÇÃO Python que executamos
FUNCOES_DISPONIVEIS = {"previsao_do_tempo": previsao_do_tempo}


def main():
    cliente = criar_cliente()
    pergunta = "Tá mais quente em Recife ou em Caruaru agora? Preciso levar guarda-chuva?"
    titulo(pergunta)

    # PASSO 2 — Pergunta + ferramentas
    info("PASSO 2: enviando pergunta e a lista de ferramentas...")
    interaction = cliente.interactions.create(
        model=MODELO,
        input=pergunta,
        tools=[DECLARACAO_CLIMA],
    )

    # O modelo respondeu com pedidos de função? (pode pedir mais de um de uma vez!)
    pedidos = [p for p in interaction.steps if p.type == "function_call"]
    if not pedidos:
        print("O modelo respondeu direto, sem usar ferramentas:\n", interaction.output_text)
        return

    # PASSO 3 — Executar cada pedido (quem executa somos NÓS)
    resultados = []
    for pedido in pedidos:
        info(f"PASSO 3: o modelo pediu {cor(pedido.name, 'magenta')}({pedido.arguments})")
        funcao = FUNCOES_DISPONIVEIS[pedido.name]
        resultado = funcao(**pedido.arguments)
        ok(f"Resultado real da API: {resultado}")

        resultados.append({
            "type": "function_result",
            "name": pedido.name,
            "call_id": pedido.id,  # liga este resultado ao pedido certo
            "result": [{"type": "text", "text": json.dumps(resultado, ensure_ascii=False)}],
        })

    # PASSO 4 — Devolver os resultados e receber a resposta final
    info("PASSO 4: devolvendo os resultados para o modelo...")
    final = cliente.interactions.create(
        model=MODELO,
        input=resultados,
        tools=[DECLARACAO_CLIMA],             # ferramentas vão em TODA chamada
        previous_interaction_id=interaction.id,  # continua a mesma conversa
    )
    print("\n" + cor("Resposta final:", "verde"))
    print(final.output_text)


if __name__ == "__main__":
    main()

---
## Módulo 4 — O agente autônomo: Guia Rec'n'Play
Criamos os arquivos do agente: dados de exemplo, ferramentas e o laço.

In [ ]:
!mkdir -p dados

In [ ]:
%%writefile dados/programacao.json
{
  "_aviso": "DADOS FICTÍCIOS DE EXEMPLO para a oficina. Substitua pela programação oficial do Rec'n'Play se quiser.",
  "evento": "Rec'n'Play (exemplo)",
  "cidade": "Recife",
  "atividades": [
    {"id": "A01", "titulo": "Abertura: o futuro da tecnologia em Pernambuco", "tipo": "palestra", "trilha": "Carreira", "dia": "quinta", "inicio": "09:00", "fim": "10:00", "local": "Palco Marco Zero", "nivel": "todos"},
    {"id": "A02", "titulo": "Python do zero em 90 minutos", "tipo": "oficina", "trilha": "Programação", "dia": "quinta", "inicio": "10:30", "fim": "12:00", "local": "Sala Frevo", "nivel": "iniciante"},
    {"id": "A03", "titulo": "Design de interfaces acessíveis", "tipo": "palestra", "trilha": "Design", "dia": "quinta", "inicio": "14:00", "fim": "15:00", "local": "Sala Maracatu", "nivel": "iniciante"},
    {"id": "A04", "titulo": "Workshop Prático: Construção de Agentes Autônomos com Gemini API e Function Calling", "tipo": "oficina", "trilha": "Inteligência Artificial", "dia": "quinta", "inicio": "13:00", "fim": "17:00", "local": "Sala Frevo", "nivel": "iniciante"},
    {"id": "A05", "titulo": "Carreira em dados: por onde começar", "tipo": "painel", "trilha": "Dados", "dia": "quinta", "inicio": "16:00", "fim": "17:00", "local": "Palco Marco Zero", "nivel": "todos"},
    {"id": "A06", "titulo": "Cloud para iniciantes: seu primeiro deploy", "tipo": "oficina", "trilha": "Cloud", "dia": "sexta", "inicio": "09:00", "fim": "11:00", "local": "Sala Capibaribe", "nivel": "iniciante"},
    {"id": "A07", "titulo": "LLMs em produção: custos, latência e qualidade", "tipo": "palestra", "trilha": "Inteligência Artificial", "dia": "sexta", "inicio": "11:00", "fim": "12:00", "local": "Palco Marco Zero", "nivel": "intermediário"},
    {"id": "A08", "titulo": "Games indies feitos no Nordeste", "tipo": "painel", "trilha": "Games", "dia": "sexta", "inicio": "14:00", "fim": "15:00", "local": "Sala Maracatu", "nivel": "todos"},
    {"id": "A09", "titulo": "Segurança para devs: OWASP na prática", "tipo": "oficina", "trilha": "Segurança", "dia": "sexta", "inicio": "14:00", "fim": "16:00", "local": "Sala Capibaribe", "nivel": "intermediário"},
    {"id": "A10", "titulo": "Prompt engineering sem mistério", "tipo": "palestra", "trilha": "Inteligência Artificial", "dia": "sexta", "inicio": "16:00", "fim": "17:00", "local": "Sala Frevo", "nivel": "iniciante"},
    {"id": "A11", "titulo": "Robótica com Arduino para todas as idades", "tipo": "oficina", "trilha": "Maker", "dia": "sábado", "inicio": "09:00", "fim": "12:00", "local": "Espaço Maker", "nivel": "iniciante"},
    {"id": "A12", "titulo": "Mulheres na tecnologia: trajetórias e redes", "tipo": "painel", "trilha": "Carreira", "dia": "sábado", "inicio": "10:00", "fim": "11:00", "local": "Palco Marco Zero", "nivel": "todos"},
    {"id": "A13", "titulo": "RAG: dando memória de documentos para a IA", "tipo": "oficina", "trilha": "Inteligência Artificial", "dia": "sábado", "inicio": "14:00", "fim": "16:00", "local": "Sala Frevo", "nivel": "intermediário"},
    {"id": "A14", "titulo": "Encerramento e Hackathon Awards", "tipo": "palestra", "trilha": "Comunidade", "dia": "sábado", "inicio": "17:00", "fim": "18:00", "local": "Palco Marco Zero", "nivel": "todos"}
  ]
}

In [ ]:
%%writefile ferramentas.py
"""
ferramentas.py — as "mãos" do agente.

Cada ferramenta tem DUAS partes:
  1. A função Python que faz o trabalho de verdade
  2. A DECLARAÇÃO (dicionário) que explica ao modelo o que ela faz

Regras de ouro que seguimos aqui (e que evitam alucinação):
  • Nunca deixar uma exceção "explodir": devolvemos {"erro": "..."} para o modelo
    entender o que deu errado e contar a verdade ao usuário.
  • Validar TUDO que o modelo manda (um CEP com 8 dígitos, um id que existe...).
  • Devolver dados objetivos e pequenos. O modelo escreve o texto bonito depois.
"""
import json
import re
from pathlib import Path

import requests

PASTA_DADOS = Path(__file__).resolve().parent / "dados"
ARQUIVO_PROGRAMACAO = PASTA_DADOS / "programacao.json"
ARQUIVO_AGENDA = PASTA_DADOS / "minha_agenda.json"
TIMEOUT = 10  # segundos. Nunca chame uma API externa sem timeout!


# ════════════════════════════════════════════════════════════════
# Funções auxiliares (não são ferramentas, o modelo não as vê)
# ════════════════════════════════════════════════════════════════
def _carregar_atividades() -> list[dict]:
    return json.loads(ARQUIVO_PROGRAMACAO.read_text(encoding="utf-8"))["atividades"]


def _normalizar(texto: str) -> str:
    """Deixa em minúsculas e sem acentos, para buscas mais tolerantes."""
    trocas = str.maketrans("áàâãéêíóôõúüç", "aaaaeeiooouuc")
    return (texto or "").lower().translate(trocas).strip()


def _ler_agenda() -> list[str]:
    if not ARQUIVO_AGENDA.exists():
        return []
    return json.loads(ARQUIVO_AGENDA.read_text(encoding="utf-8"))


def _salvar_agenda(ids: list[str]):
    ARQUIVO_AGENDA.write_text(json.dumps(ids, ensure_ascii=False, indent=2), encoding="utf-8")


def _conflita(a: dict, b: dict) -> bool:
    """Duas atividades no mesmo dia se sobrepõem no horário?"""
    return a["dia"] == b["dia"] and a["inicio"] < b["fim"] and b["inicio"] < a["fim"]


# ════════════════════════════════════════════════════════════════
# FERRAMENTA 1 — Consultar a programação (dados locais)
# ════════════════════════════════════════════════════════════════
def consultar_programacao(termo: str = "", dia: str = "", trilha: str = "", a_partir_das: str = "") -> dict:
    atividades = _carregar_atividades()
    encontradas = []
    for a in atividades:
        if termo and _normalizar(termo) not in _normalizar(a["titulo"] + " " + a["trilha"]):
            continue
        if dia and _normalizar(dia) not in _normalizar(a["dia"]):
            continue
        if trilha and _normalizar(trilha) not in _normalizar(a["trilha"]):
            continue
        if a_partir_das and a["inicio"] < a_partir_das:
            continue
        encontradas.append(a)
    return {"total": len(encontradas), "atividades": encontradas}


DECL_CONSULTAR_PROGRAMACAO = {
    "type": "function",
    "name": "consultar_programacao",
    "description": (
        "Busca atividades na programação oficial do evento. É a ÚNICA fonte confiável sobre "
        "palestras, oficinas, horários e salas. Todos os filtros são opcionais e podem ser combinados. "
        "Sem filtros, retorna a programação completa."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "termo": {"type": "string", "description": "Palavra-chave no título ou trilha, ex.: 'python', 'IA', 'agentes'"},
            "dia": {"type": "string", "enum": ["quinta", "sexta", "sábado"], "description": "Dia do evento"},
            "trilha": {"type": "string", "description": "Trilha temática, ex.: 'Inteligência Artificial', 'Cloud', 'Carreira'"},
            "a_partir_das": {"type": "string", "description": "Horário mínimo de início no formato HH:MM"},
        },
    },
}


# ════════════════════════════════════════════════════════════════
# FERRAMENTA 2 — Clima (API externa: Open-Meteo, sem chave)
# ════════════════════════════════════════════════════════════════
def previsao_do_tempo(cidade: str) -> dict:
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": cidade, "count": 1, "language": "pt", "countryCode": "BR"},
            timeout=TIMEOUT,
        ).json()
        if not geo.get("results"):
            return {"erro": f"Cidade '{cidade}' não encontrada no Brasil."}
        lugar = geo["results"][0]

        dados = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lugar["latitude"],
                "longitude": lugar["longitude"],
                "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                "forecast_days": 3,
                "timezone": "America/Recife",
            },
            timeout=TIMEOUT,
        ).json()
    except requests.RequestException as e:
        return {"erro": f"Serviço de clima indisponível no momento: {e.__class__.__name__}"}

    atual, dias = dados["current"], dados["daily"]
    return {
        "cidade": f"{lugar['name']} - {lugar.get('admin1', '')}",
        "agora": {
            "temperatura_c": atual["temperature_2m"],
            "umidade_pct": atual["relative_humidity_2m"],
            "chuva_mm": atual["precipitation"],
            "vento_kmh": atual["wind_speed_10m"],
            "medido_em": atual["time"],
        },
        "proximos_dias": [
            {"data": d, "max_c": mx, "min_c": mn, "chance_chuva_pct": pc}
            for d, mx, mn, pc in zip(dias["time"], dias["temperature_2m_max"],
                                     dias["temperature_2m_min"], dias["precipitation_probability_max"])
        ],
        "fonte": "open-meteo.com",
    }


DECL_PREVISAO_DO_TEMPO = {
    "type": "function",
    "name": "previsao_do_tempo",
    "description": "Clima atual e previsão para os próximos 3 dias de uma cidade brasileira (temperatura e chance de chuva).",
    "parameters": {
        "type": "object",
        "properties": {"cidade": {"type": "string", "description": "Nome da cidade, ex.: Recife"}},
        "required": ["cidade"],
    },
}


# ════════════════════════════════════════════════════════════════
# FERRAMENTA 3 — Endereço por CEP (API externa: ViaCEP, sem chave)
# ════════════════════════════════════════════════════════════════
def buscar_endereco_por_cep(cep: str) -> dict:
    somente_digitos = re.sub(r"\D", "", cep or "")
    if len(somente_digitos) != 8:  # validação ANTES de chamar a API
        return {"erro": "CEP inválido. Um CEP tem exatamente 8 dígitos, ex.: 50030-230."}
    try:
        dados = requests.get(f"https://viacep.com.br/ws/{somente_digitos}/json/", timeout=TIMEOUT).json()
    except requests.RequestException as e:
        return {"erro": f"Serviço de CEP indisponível: {e.__class__.__name__}"}
    if dados.get("erro"):
        return {"erro": f"CEP {somente_digitos} não existe na base dos Correios."}
    return {k: dados.get(k) for k in ("cep", "logradouro", "bairro", "localidade", "uf")}


DECL_BUSCAR_ENDERECO_POR_CEP = {
    "type": "function",
    "name": "buscar_endereco_por_cep",
    "description": "Converte um CEP brasileiro em endereço (rua, bairro, cidade e UF).",
    "parameters": {
        "type": "object",
        "properties": {"cep": {"type": "string", "description": "CEP com ou sem hífen, ex.: 50030-230"}},
        "required": ["cep"],
    },
}


# ════════════════════════════════════════════════════════════════
# FERRAMENTA 4 — Adicionar à minha agenda (uma AÇÃO que muda o mundo)
# ════════════════════════════════════════════════════════════════
def adicionar_na_minha_agenda(id_atividade: str) -> dict:
    atividades = {a["id"]: a for a in _carregar_atividades()}
    id_atividade = (id_atividade or "").strip().upper()

    # Guardrail anti-alucinação: só aceitamos ids que EXISTEM
    if id_atividade not in atividades:
        return {"erro": f"Não existe atividade com id '{id_atividade}'. Consulte a programação primeiro."}

    agenda = _ler_agenda()
    if id_atividade in agenda:
        return {"ok": False, "mensagem": "Essa atividade já está na sua agenda."}

    nova = atividades[id_atividade]
    conflitos = [atividades[i] for i in agenda if i in atividades and _conflita(nova, atividades[i])]
    if conflitos:  # Regra de negócio: não deixamos marcar duas coisas no mesmo horário
        return {
            "ok": False,
            "mensagem": "Conflito de horário. Nada foi adicionado.",
            "conflita_com": [{"id": c["id"], "titulo": c["titulo"], "inicio": c["inicio"], "fim": c["fim"]} for c in conflitos],
        }

    agenda.append(id_atividade)
    _salvar_agenda(agenda)
    return {"ok": True, "adicionada": nova, "total_na_agenda": len(agenda)}


DECL_ADICIONAR_NA_MINHA_AGENDA = {
    "type": "function",
    "name": "adicionar_na_minha_agenda",
    "description": (
        "Salva uma atividade na agenda pessoal do usuário. Só use quando o usuário pedir claramente "
        "para adicionar/salvar/marcar. Exige o id exato (ex.: 'A04') obtido em consultar_programacao. "
        "Recusa automaticamente se houver conflito de horário."
    ),
    "parameters": {
        "type": "object",
        "properties": {"id_atividade": {"type": "string", "description": "Id da atividade, ex.: A04"}},
        "required": ["id_atividade"],
    },
}


# ════════════════════════════════════════════════════════════════
# FERRAMENTA 5 — Ver minha agenda
# ════════════════════════════════════════════════════════════════
def ver_minha_agenda() -> dict:
    atividades = {a["id"]: a for a in _carregar_atividades()}
    itens = [atividades[i] for i in _ler_agenda() if i in atividades]
    ordem_dias = {"quinta": 0, "sexta": 1, "sábado": 2}
    itens.sort(key=lambda a: (ordem_dias.get(a["dia"], 9), a["inicio"]))
    return {"total": len(itens), "atividades": itens}


DECL_VER_MINHA_AGENDA = {
    "type": "function",
    "name": "ver_minha_agenda",
    "description": "Lista as atividades que o usuário já salvou na agenda pessoal, em ordem cronológica.",
    "parameters": {"type": "object", "properties": {}},
}


# ════════════════════════════════════════════════════════════════
# Registro: juntamos tudo num lugar só
# ════════════════════════════════════════════════════════════════
DECLARACOES = [
    DECL_CONSULTAR_PROGRAMACAO,
    DECL_PREVISAO_DO_TEMPO,
    DECL_BUSCAR_ENDERECO_POR_CEP,
    DECL_ADICIONAR_NA_MINHA_AGENDA,
    DECL_VER_MINHA_AGENDA,
]

FUNCOES = {
    "consultar_programacao": consultar_programacao,
    "previsao_do_tempo": previsao_do_tempo,
    "buscar_endereco_por_cep": buscar_endereco_por_cep,
    "adicionar_na_minha_agenda": adicionar_na_minha_agenda,
    "ver_minha_agenda": ver_minha_agenda,
}


def executar_ferramenta(nome: str, argumentos: dict) -> dict:
    """Executa a ferramenta pedida pelo modelo, sem nunca quebrar o agente."""
    funcao = FUNCOES.get(nome)
    if funcao is None:
        return {"erro": f"Ferramenta '{nome}' não existe."}
    try:
        return funcao(**(argumentos or {}))
    except TypeError as e:  # o modelo mandou um argumento com nome errado
        return {"erro": f"Argumentos inválidos para {nome}: {e}"}
    except Exception as e:  # qualquer outra falha vira informação, não crash
        return {"erro": f"Falha inesperada em {nome}: {e}"}

In [ ]:
%%writefile agente.py
"""
agente.py — o "cérebro + laço" do agente autônomo.

O que torna isto um AGENTE (e não só um chatbot)?
  1. Ele tem um OBJETIVO (a pergunta do usuário)
  2. Ele tem FERRAMENTAS (ferramentas.py)
  3. Ele roda em LAÇO: pensa → pede ferramenta → recebe resultado → pensa de novo...
     até decidir que já tem o suficiente para responder.

      ┌─────────────┐   function_call   ┌──────────────┐
      │   Gemini    │ ────────────────▶ │  Nosso código │
      │  (decide)   │ ◀──────────────── │  (executa)    │
      └─────────────┘  function_result  └──────────────┘
             │  sem mais pedidos?
             ▼
      resposta final ao usuário

Proteções contra "agente descontrolado" e alucinação:
  • MAX_PASSOS: limite de voltas no laço (evita loops infinitos e contas altas)
  • INSTRUCOES: regras claras de só afirmar o que veio das ferramentas
  • Erros das ferramentas voltam como dados, para o modelo admitir a falha
  • Registro (log) de cada passo, para você auditar o que aconteceu
"""
import json
import sys
from pathlib import Path
from typing import List, Literal

from pydantic import BaseModel, Field

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from comum import MODELO, cor  # noqa: E402
from ferramentas import DECLARACOES, executar_ferramenta  # noqa: E402

MAX_PASSOS = 8

INSTRUCOES = """
Você é o Guia Rec'n'Play, um assistente que ajuda participantes do evento em Recife.

REGRAS OBRIGATÓRIAS:
1. Informações sobre programação, horários, salas, clima e endereços SÓ podem vir das ferramentas.
   Nunca invente atividades, horários, ids, temperaturas ou endereços.
2. Se uma ferramenta devolver "erro" ou nenhum resultado, diga isso com honestidade
   e sugira o que o usuário pode fazer.
3. Se faltar informação para agir (ex.: qual atividade adicionar), PERGUNTE antes de agir.
4. Só use adicionar_na_minha_agenda quando o usuário pedir claramente.
5. Você pode encadear várias ferramentas para resolver uma tarefa complexa.
6. Responda em português do Brasil, de forma curta e organizada. Cite ids das atividades (ex.: A04).
"""


class RespostaEstruturada(BaseModel):
    """Formato final opcional, útil para exibir em um app ou salvar em banco."""
    resposta: str = Field(description="Resposta final para o usuário, em português")
    ferramentas_usadas: List[str] = Field(description="Nomes das ferramentas usadas para chegar na resposta")
    ids_atividades_citadas: List[str] = Field(description="Ids de atividades mencionadas, ex.: ['A04']")
    confianca: Literal["alta", "media", "baixa"] = Field(
        description="alta se tudo veio das ferramentas; baixa se faltou dado ou houve erro"
    )
    proxima_acao_sugerida: str = Field(description="Uma sugestão curta do que o usuário pode fazer em seguida")


class Agente:
    def __init__(self, cliente, verboso: bool = True):
        self.cliente = cliente
        self.verboso = verboso
        self.id_conversa = None  # guarda o id da última interação (memória da conversa)
        self.ferramentas_usadas: list[str] = []

    # ───────────────────────── utilitário de log ─────────────────────────
    def _log(self, texto: str):
        if self.verboso:
            print(cor(f"   {texto}", "cinza"))

    # ───────────────────────── chamada ao modelo ─────────────────────────
    def _chamar_modelo(self, entrada):
        parametros = dict(
            model=MODELO,
            input=entrada,
            tools=DECLARACOES,            # interaction-scoped: vai em TODA chamada
            system_instruction=INSTRUCOES,  # idem
        )
        if self.id_conversa:
            parametros["previous_interaction_id"] = self.id_conversa
        interaction = self.cliente.interactions.create(**parametros)
        self.id_conversa = interaction.id
        return interaction

    # ───────────────────────── O LAÇO DO AGENTE ─────────────────────────
    def perguntar(self, pergunta: str) -> str:
        self.ferramentas_usadas = []
        interaction = self._chamar_modelo(pergunta)

        for passo in range(1, MAX_PASSOS + 1):
            pedidos = [s for s in interaction.steps if s.type == "function_call"]

            # Condição de parada: o modelo não pediu mais nada → terminou
            if not pedidos:
                return interaction.output_text

            # Executa TODOS os pedidos deste passo (podem vir vários em paralelo)
            resultados = []
            for pedido in pedidos:
                self._log(f"🔧 passo {passo}: {pedido.name}({json.dumps(pedido.arguments, ensure_ascii=False)})")
                resultado = executar_ferramenta(pedido.name, pedido.arguments)
                self.ferramentas_usadas.append(pedido.name)
                resumo = json.dumps(resultado, ensure_ascii=False)
                self._log(f"   ↳ {resumo[:160]}{'…' if len(resumo) > 160 else ''}")
                resultados.append({
                    "type": "function_result",
                    "name": pedido.name,
                    "call_id": pedido.id,
                    "result": [{"type": "text", "text": json.dumps(resultado, ensure_ascii=False)}],
                })

            # Devolve os resultados e deixa o modelo decidir o próximo passo
            interaction = self._chamar_modelo(resultados)

        # Se chegou aqui, estourou o limite de passos
        return ("Desculpe, não consegui concluir essa tarefa dentro do limite de "
                f"{MAX_PASSOS} passos. Tente dividir o pedido em partes menores.")

    # ───────────── Saída estruturada da última resposta (bônus) ─────────────
    def resumir_em_json(self) -> RespostaEstruturada:
        """Pede ao modelo para reescrever a ÚLTIMA resposta no formato RespostaEstruturada.

        Usamos previous_interaction_id para ele enxergar tudo o que aconteceu
        (inclusive os resultados das ferramentas), e response_format para
        garantir um JSON válido.
        """
        interaction = self.cliente.interactions.create(
            model=MODELO,
            previous_interaction_id=self.id_conversa,
            input=(
                "Reescreva sua última resposta no formato JSON pedido. "
                f"Ferramentas usadas neste turno: {sorted(set(self.ferramentas_usadas))}."
            ),
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": RespostaEstruturada.model_json_schema(),
            },
        )
        return RespostaEstruturada.model_validate_json(interaction.output_text)

### Conversando com o agente
Rode uma célula por vez e leia o log `🔧 passo` para ver as decisões.

In [ ]:
import importlib, ferramentas, agente as mod_agente
importlib.reload(ferramentas); importlib.reload(mod_agente)
from pathlib import Path
Path("dados/minha_agenda.json").unlink(missing_ok=True)   # começa com agenda vazia

guia = mod_agente.Agente(cliente)
print(guia.perguntar("Quais oficinas de IA tem na sexta?"))

In [ ]:
print(guia.perguntar("Vai chover em Recife nos próximos dias?"))

In [ ]:
print(guia.perguntar("Me mostra oficinas para iniciantes na quinta e adiciona a de agentes na minha agenda"))

In [ ]:
print(guia.perguntar("Adiciona também a A03"))   # conflito de horário!

In [ ]:
print(guia.perguntar("Adiciona a Z99"))          # id inexistente: não pode inventar

In [ ]:
print(guia.perguntar("Monte meu sábado com algo de IA e algo de carreira, sem conflito, e diga se levo guarda-chuva"))

### Bônus: a última resposta como dado estruturado

In [ ]:
print(guia.resumir_em_json().model_dump_json(indent=2))

### Faça sua própria pergunta 👇

In [ ]:
print(guia.perguntar("ESCREVA SUA PERGUNTA AQUI"))

---
## 🏆 Desafios
Edite a célula do `ferramentas.py`, rode de novo e recrie o agente.
- ⭐ `remover_da_minha_agenda(id_atividade)`
- ⭐⭐ `cotacao_moeda(moeda)` com https://economia.awesomeapi.com.br/json/last/USD-BRL
- ⭐⭐⭐ Adicione `{"type": "google_search"}` em `DECLARACOES`

**O modelo decide. O código executa. Os dados provam.** 💙 GDG Recife